# 💬 Aula 1 — Pipeline de Processamento de Texto

Ao final do notebook, você deverá ser capaz de:

- Implementar (em Python) um pipeline básico de processamento de texto como um fluxo de engenharia.
  1. **Higienização** do texto com expressões regulares  
  2. **Normalização** (minúsculas e acentos)
  3. **Tokenização**
  4. **Stopwords** (dependente da tarefa)
  5. **Redução morfológica** (discussão: stemming vs lematização)
- Produzir a saída final do pipeline pronta para a etapa de extração de características (BoW/TF-IDF).

> Observação: nesta aula, o foco é entender e implementar as transformações. Bibliotecas completas (spaCy, NLTK) serão introduzidas posteriormente.


---

## 🧰 0) Preparação do ambiente

Este notebook foi construído para funcionar apenas com bibliotecas padrão do Python.


In [41]:
# Para operações com expressões regulares
import re
# Para normalização de caracteres Unicode (como remoção de acentos)
import unicodedata
# Para anotações de tipo, melhorando a legibilidade e robustez do código
from typing import List, Dict, Any

---

## 📌 1) Texto de exemplo do pipeline



In [42]:
texto_bruto = "O produto (http://exemplo.com) é bom ou ruim, mas não é <p title='Destaque'>perfeito</p> e tem defeitos"
print("Texto bruto:", texto_bruto)


Texto bruto: O produto (http://exemplo.com) é bom ou ruim, mas não é <p title='Destaque'>perfeito</p> e tem defeitos


---

## 🧱 2) Etapa 1 — Higienização (RegEx)

**Objetivo:** remover ruídos estruturais típicos (URLs, HTML, emojis, caracteres especiais, pontuação excessiva).

> Comentário: diferentes soluções podem manter ou remover certos sinais. O importante é que a saída esteja **sem URL/emoji** e com pontuação “domada”.


In [43]:
def higienizar(texto: str) -> str:
    # remove marcações HTML
    texto = re.sub(r"<[^>]+>", " ", texto)
    # remove URLs
    texto = re.sub(r"http\S+|www\.\S+", "", texto)
    # remove pontuação e símbolos (mantém letras/números/_ e espaços)
    texto = re.sub(r"[^\w\s]", " ", texto, flags=re.UNICODE)
    # normaliza espaços
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

texto_limpo = higienizar(texto_bruto)
print("Após higienização: ", texto_limpo)

Após higienização:  O produto é bom ou ruim mas não é perfeito e tem defeitos


---

## 🔄 3) Etapa 2 — Normalização

**Objetivo:** reduzir variância do vocabulário (minúsculas e acentos).

Decisões típicas:
- aplicar `lower()`
- remover acentos para unificar grafias (ex.: "rápido" → "rapido")

⚠️ Observação: remover acentos pode ser inadequado em algumas tarefas (ex.: correção ortográfica), mas é útil em muitos cenários práticos.


In [44]:
PALAVRAS_PRESERVADAS = {"é"}

def normalizar(texto: str, remover_acentos: bool = True) -> str:
        texto = texto.lower()
        tokens = texto.split()

        if remover_acentos:
            tokens_normalizados = []
            for token in tokens:
                if token in PALAVRAS_PRESERVADAS:
                    tokens_normalizados.append(token)
                else:
                    token = unicodedata.normalize("NFD", token)
                    token = "".join(c for c in token if not unicodedata.combining(c))
                    tokens_normalizados.append(token)

            texto = " ".join(tokens_normalizados)

        texto = re.sub(r"\s+", " ", texto).strip()
        return texto

texto_normalizado = normalizar(texto, remover_acentos=True)
print("Após normalização:", texto_normalizado)

Após normalização: nao gostei... o produto veio com defeito 😡 http://exemplo.com


---

## ✂️ 4) Etapa 3 — Tokenização

**Objetivo:** quebrar o texto em unidades menores (tokens).

Utilizaremos uma tokenização por palavras extremamente simples (`split()`), por ser transparente e suficiente para fixar o conceito.


In [45]:
def tokenizar(texto: str) -> List[str]:
    return texto.split()

tokens = tokenizar(texto_normalizado)
print("Tokens:", tokens)


Tokens: ['nao', 'gostei...', 'o', 'produto', 'veio', 'com', 'defeito', '😡', 'http://exemplo.com']


---

## 🚫 5) Etapa 4 — Stopwords

**Objetivo:** remover palavras muito frequentes que podem ser ruído para algumas tarefas.

Ponto didático central: **não existe decisão universal.**  
Em **análise de sentimentos**, palavras como **"nao"** podem ser vitais.  
Em **classificação de tópicos**, muitas stopwords podem ser ruído.

Abaixo usaremos uma lista mínima apenas para exercício.


In [46]:
STOPWORDS_MIN = {
    "o", "a", "os", "as",
    "de", "do", "da", "dos", "das",
    "e", "em", "no", "na", "nao", "nos", "nas",
    "com", "por", "para", "um", "uma", "uns", "umas"
}

def remover_stopwords(tokens: List[str], manter: set = None) -> List[str]:
    manter = manter or set()
    saida = []
    for t in tokens:
        if t in manter:
            saida.append(t)
        elif t not in STOPWORDS_MIN:
            saida.append(t)
    return saida

tokens_filtrados = remover_stopwords(tokens, manter={"nao"})
print("Sem stopwords", tokens_filtrados)


Sem stopwords ['nao', 'gostei...', 'produto', 'veio', 'defeito', '😡', 'http://exemplo.com']


---

## 🌱 6) Etapa 5 — Redução morfológica (conceitual)

Nesta aula, o foco é compreender o trade-off:

- **Stemming**: rápido/heurístico (pode “quebrar” palavras)
- **Lematização**: mais precisa/gramatical (exige recursos linguísticos)

Como o Python padrão não oferece lematização pronta, faremos:

1. **Discussão conceitual**
2. Um exemplo didático mínimo via dicionário manual (apenas para demonstrar o efeito)


In [47]:
# Exemplo didático mínimo (não é um lematizador real)
LEMA_MANUAL = {
    "gostei": "gostar",
    "veio": "vir",
    "defeito": "defeito",  # já está em forma base
    "produto": "produto",
    "nao": "nao"
}

def lematizar(tokens: List[str]) -> List[str]:
    return [LEMA_MANUAL.get(t, t) for t in tokens]

tokens_reduzidos = lematizar(tokens_filtrados)
print("Redução morfológica:", tokens_reduzidos)


Redução morfológica: ['nao', 'gostei...', 'produto', 'vir', 'defeito', '😡', 'http://exemplo.com']


> Comentário: o lema de "veio" como "vir" é correto linguisticamente, mas, dependendo da tarefa, pode-se optar por manter "veio".


---

## 🧩 7) Saída final do pipeline

Agora consolidaremos o pipeline como uma função de engenharia (entrada → transformações → saída).


In [48]:
def pipeline_pln_basico(texto_bruto: str, tarefa: str = "sentimentos") -> Dict[str, Any]:
    # 1) Higienização
    texto_limpo = higienizar(texto_bruto)

    # 2) Normalização
    texto_normalizado = normalizar(texto_limpo, remover_acentos=True)

    # 3) Tokenização
    tokens = tokenizar(texto_normalizado)

    # 4) Stopwords
    if tarefa == "sentimentos":
        tokens_filtrados = remover_stopwords(tokens, manter={"nao"})
    else:
        tokens_filtrados = remover_stopwords(tokens, manter=set())

    # 5) Redução morfológica
    tokens_reduzidos = lematizar(tokens_filtrados)

    return {
        "texto_bruto": texto_bruto,
        "texto_limpo": texto_limpo,
        "texto_normalizado": texto_normalizado,
        "tokens": tokens,
        "tokens_filtrados": tokens_filtrados,
        "tokens_reduzidos": tokens_reduzidos
    }

texto = "<p title='resposta'>Produto ok</p>, mas a embalagem estava DANIFICADA!!!"
texto = "Não gostei... o produto veio com defeito 😡 http://exemplo.com"
resultado = pipeline_pln_basico(texto, tarefa="sentimentos")
for k, v in resultado.items():
    print(f"{k:>15}: {v}")


    texto_bruto: Não gostei... o produto veio com defeito 😡 http://exemplo.com
    texto_limpo: Não gostei o produto veio com defeito
texto_normalizado: nao gostei o produto veio com defeito
         tokens: ['nao', 'gostei', 'o', 'produto', 'veio', 'com', 'defeito']
tokens_filtrados: ['nao', 'gostei', 'produto', 'veio', 'defeito']
tokens_reduzidos: ['nao', 'gostar', 'produto', 'vir', 'defeito']
